# Goal: obtain structural parameters from simulated data 

In [8]:
import import_ipynb
import importlib
import BLP_functions as f 
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from numpy.linalg import inv
import builtins

np.set_printoptions(precision=3, suppress=True)
np.set_printoptions(legacy='1.13')

tolerance = 0.01
instrument_choice = 'instrument_char'

#structural parameters:  UNKNOWN - TO BE ESTIMATED
nonlinear_guess = 5
linear_guess = [0,0] 
n_consumer_sim = 1000 #number of consumer to be simulated

market_data = pd.read_csv('data.csv')

n_market = np.unique(market_data['market_id']).size
n_product = np.sum(market_data['market_id'] == 0)
print(n_market, n_product)

#instrument_char is usually good

importlib.reload(f)

200 7
changed third


ModuleSpec(name='BLP_functions', loader=<import_ipynb.NotebookLoader object at 0x10dddc440>, origin='BLP_functions.ipynb')

A simple first practice for BLP 

- Outer loop: estimating the sigma parameter, which is the variance of the demand coefficient for product characteristics beta 
- Inner loop: contraction mapping and calculating linear parameters which are demand coefficient for price (alpha, which assumed to be homogenous), and beta (which is heterogeneous due to sigma)

In [9]:
# # checking the estimation functions 
market_i = market_data[market_data['market_id'] == 2]
observed_share = market_i[market_i['market_id'] == 2]['share'].values

sim_utilities = np.array(f.utility_gen(linear_guess, nonlinear_guess, market_i['prod_char'], market_i['price'], n_consumer_sim))

print(market_i[market_i['market_id'] == 2]['true_delta'].values)
print("converged delta", f.contraction_mapping(sim_utilities, observed_share, tolerance))

np.corrcoef(market_data[instrument_choice], market_data['price'])[0, 1]

import statsmodels
from statsmodels.api import OLS, add_constant
X = add_constant(market_data[instrument_choice].values)
y = market_data['price']
print(OLS(y, X).fit().summary())
# # Let's try to check of contraction mapping recovered the true deltas 


[ 5.99  -4.079 -4.431 -1.267  5.031 -7.276  6.033]
converged delta [ 10.006  -3.647 -26.916  -3.867   9.435 -26.173  10.198]
                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.213
Model:                            OLS   Adj. R-squared:                  0.212
Method:                 Least Squares   F-statistic:                     378.5
Date:                Thu, 03 Jul 2025   Prob (F-statistic):           8.55e-75
Time:                        14:17:58   Log-Likelihood:                -1818.8
No. Observations:                1400   AIC:                             3642.
Df Residuals:                    1398   BIC:                             3652.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------

In [ ]:
# # Main GMM code:
def objective_function(sigma_guess, input_dataset, parameter_guess, tolerance, n_consumer_sim): #return loss value which we want to minimize 
    delta_array = f.mean_utility(input_dataset, sigma_guess, parameter_guess, tolerance, n_consumer_sim)
    alpha_hat, beta_hat = f.estimate_parameter(input_dataset, instrument_choice, delta_array)
    # print(alpha_hat, beta_hat)
    # calculating residual (error terms)
    x = input_dataset['prod_char'].values
    p = input_dataset['price'].values
    delta_array = f.mean_utility(input_dataset, sigma_guess, [alpha_hat, beta_hat], tolerance, n_consumer_sim)
    # the epsilon term ~ unobserved heterogenerous utility
    residual = delta_array - (beta_hat * x + alpha_hat * p)
    # print(residual, len(residual))
    Z = input_dataset[[instrument_choice,'price']].values
    # print(Z[:30], residual[:30])
    g = Z.T @ residual/ len(residual)
    loss = g.T @ g
    # print("loss value", loss, "for sigma equals", sigma_guess)
    return g.T @ g #loss function of the residuals 

print(objective_function(nonlinear_guess, market_data, linear_guess, tolerance, n_consumer_sim))
def main(instrument_choice, opt_method, nonlinear_guess, linear_guess, market_data, tolerance, n_consumer_sim):
    print("We use the instrument", instrument_choice, "with method", opt_method)
    result = minimize(
        objective_function, 
        x0 = nonlinear_guess, 
        args=(market_data, linear_guess, tolerance, n_consumer_sim), 
        method= opt_method, 
        options= {'disp': False}
    )
    sigma_estimated = result.x
    print(result.success)
    delta_final = f.mean_utility(market_data, sigma_estimated, linear_guess, tolerance, n_consumer_sim)
    linear_estimated = f.estimate_parameter(market_data, instrument_choice, delta_final)
    print("Estimated nonlinear:", sigma_estimated, "and linear:", linear_estimated) 


main('instrument_char', 'Powell', nonlinear_guess, linear_guess, market_data, tolerance, n_consumer_sim)
# main('instrument_cost', 'Powell',nonlinear_guess, linear_guess, market_data, tolerance, n_consumer_sim)
# main('instrument_price', 'Powell',nonlinear_guess, linear_guess, market_data, tolerance, n_consumer_sim)


123514.528766
We use the instrument instrument_char with method Nelder-Mead


KeyboardInterrupt: 

In [ ]:
# # MULTI-PROCESSING VERSION

# # # Main GMM code:
# def objective_function(sigma_guess, input_dataset, parameter_guess, tolerance, n_consumer_sim): #return loss value which we want to minimize 
#     delta_array = f.mean_utility(input_dataset, sigma_guess, parameter_guess, tolerance, n_consumer_sim)
#     alpha_hat, beta_hat = f.estimate_parameter(input_dataset, instrument_choice, delta_array)
#     # print(alpha_hat, beta_hat)
#     # calculating residual (error terms)
#     x = input_dataset['prod_char'].values
#     p = input_dataset['price'].values
#     # the epsilon term ~ unobserved heterogenerous utility
#     residual = delta_array - (beta_hat * x + alpha_hat * p)
#     # print(residual)
#     Z = input_dataset[[instrument_choice,'price']].values
#     g = Z.T @ residual
#     return g.T @ g #loss function of the residuals 

# print(objective_function(nonlinear_guess, market_data, linear_guess, tolerance, n_consumer_sim))

# result = minimize(
#     objective_function, 
#     x0 = nonlinear_guess, 
#     args=(market_data, linear_guess, tolerance, n_consumer_sim), 
#     method= 'Nelder-Mead', 
#     options= {'disp': True}
# )

# print("found!")
# sigma_estimated = result.x
# print("Estimated sigma:", sigma_estimated) 
# delta_final = f.mean_utility(market_data, sigma_estimated, linear_guess, tolerance, n_consumer_sim)
# linear_estimated = f.estimate_parameter(market_data, instrument_choice, delta_final)

# print("Estimated parameter newly found to be:", linear_estimated)